# Full Arabic Vocalization with Mishkal

This notebook demonstrates the full-vocalization stage. The production implementation is in `scripts/vocalize_with_mishkal.py`.

Unlike the original exploratory notebook, this workflow preserves sentence alignment, writes structured TSV output, logs failures, avoids append-mode duplication, and records a run summary.

## Input

The expected input is `accepted_pairs.tsv` from the filtering stage. It must contain a `text_ar` column. The output adds `arabic_vocalized` while preserving all original columns.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_PATH = REPO_ROOT / "data" / "processed" / "filter_demo" / "accepted_pairs.tsv"
OUTPUT_PATH = REPO_ROOT / "data" / "processed" / "vocalized_demo.tsv"
LIMIT = 100  # Set to None for all accepted pairs.
INPUT_PATH, OUTPUT_PATH

In [ ]:
command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "vocalize_with_mishkal.py"),
    "--input", str(INPUT_PATH),
    "--output", str(OUTPUT_PATH),
    "--overwrite",
]
if LIMIT is not None:
    command.extend(["--limit", str(LIMIT)])

print(" ".join(command))
# Uncomment to execute:
# subprocess.run(command, check=True)

## Inspect outputs

Rows that Mishkal cannot process remain aligned with an empty generated field, and the corresponding exception is written to a JSONL error file.

In [ ]:
import csv

if OUTPUT_PATH.exists():
    with OUTPUT_PATH.open("r", encoding="utf-8", newline="") as handle:
        preview = list(csv.DictReader(handle, delimiter="\t"))[:5]
    preview
else:
    print("Run the command cell first; no vocalized output exists yet.")

In [ ]:
summary_path = OUTPUT_PATH.with_suffix(".summary.json")
if summary_path.exists():
    json.loads(summary_path.read_text(encoding="utf-8"))
else:
    print("No summary file exists yet.")